In [4]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks
from scipy.interpolate import interp1d
sns.set_theme()

In [3]:
def get_force_imu_data(filepath, fs_imu=2000):
    """
    load force data (computed by nexus), imu data and extract sampling rates
    :param filepath: current csv file path
    :param fs_imu: int with IMU sampling rate, default = 2000 Hz
    :return force_df: df with force data
    :return fs_force: int with force sampling rate
    :return imu_df: df with force data
    :return fs_imu: int with force sampling rate
    """
    df = pd.read_csv(filepath, sep=',', header=[3], low_memory=False)
    # get force data
    ff = df.iloc[df[df['Frame']=='Frame'].index[0]+2:, :]
    ff.columns = df.iloc[df[df['Frame'] == 'Frame'].index[0], :].values.tolist()
    force_df = ff[['Frame', 'Sub Frame', 'Fx','Fy','Fz','Mx','My','Mz','Cx','Cy','Cz']].astype('float')
    force_df.reset_index(inplace=False)
    # get fs_force
    fs_force = int(len(force_df)/(df[df['Frame']=='Frame'].index[0]-4)*fs_imu) # number of frames force/number of frames IMU * fs IMU (2000 Hz)

    # get imu data
    imu_df = df.iloc[1:df[df['Frame'] == 'Devices'].index[0], :].astype('float')
    if fs_force != fs_imu:
        imu_df = imu_df[::fs_imu//fs_force]
    imu_df.reset_index(inplace=False)

    return force_df, fs_force, imu_df, fs_imu

In [5]:
def getyAxis(acc):
    # Mean for each sensor axis
    accMean = np.mean(acc, axis=0)

    # Mean normalized by the Euclidean norm of the mean
    yAxis = accMean / np.linalg.norm(accMean)

    return yAxis.reshape(-1, 1)

In [6]:
def getzAxis(gyro, direction):
    # Use here the transposed version to have same notation as in matlab code
    gyro = np.transpose(gyro)

    # Mean for each sensor axis
    gyroMean = np.mean(gyro, axis=1)

    # Mean free signal for each sensor axis
    gyroMeanFree = gyro - gyroMean[:, None]

    # SVD
    u, _, _ = np.linalg.svd(gyroMeanFree)

    # First eigenvector is approximately parallel to the axis of rotation
    zAxis = u[:, 0]

    # Project all samples on zAxis to estimate total angle
    theta = np.sum(np.transpose(gyro) @ zAxis)

    # Correct sign of zAxis
    zAxis = zAxis * np.sign(theta) * np.sign(direction)

    return zAxis.reshape(-1, 1) 

In [7]:
def wahba(v1, v2, w1, w2):
    """This function implements Wahba's problem for n = 2 and a_i = 1.
    See https://en.wikipedia.org/wiki/Wahba%27s_problem
    """

    # Obtain matrix B
    B = w1 @ np.transpose(v1) + w2 @ np.transpose(v2)

    # Perform SVD
    U, S, V_transposed = np.linalg.svd(B)

    # Compute rotation
    M = np.diag([1, 1, np.linalg.det(U) * np.linalg.det(V_transposed)])
    R = U @ M @ V_transposed

    return R

In [9]:
filepath = r'D:\Salzburg\julian_cutting_wedges\221122_pi_test\vicon_imu_cal.csv'
#force_df, fs_force, imu_df, fs_imu = get_force_imu_data(filepath, fs_imu=2000)
df = pd.read_csv(filepath, sep=',', header=[3], low_memory=False)

In [12]:
df.columns

Index(['Frame', 'Sub Frame', 'X', 'Y', 'Z', 'X.1', 'Y.1', 'Z.1', 'X.2', 'Y.2',
       ...
       'Z.46', 'X.47', 'Y.47', 'Z.47', 'X.48', 'Y.48', 'Z.48', 'X.49', 'Y.49',
       'Z.49'],
      dtype='object', length=152)

In [63]:
a = pd.Series([1., 1.1, 1.2, 2, 2.1, 2.2, 3, 3.1, 3.2])

In [20]:
a[::3]

0    1.0
3    2.0
6    3.0
dtype: float64

In [87]:
def downsample1d(array, fs_high, fs_low):
    # check if devidable
    if fs_high%fs_low != 0:
        print('sampling frequencies cannot be devided without remainder')
        return
    elif fs_high>len(array):
        print('sampling frequency does not match array lenght')
        return
    else:
        conversion = fs_high//fs_low
        array_down = np.empty(len(array)//conversion)
        for i in range(0, len(array), conversion):
            # print(i, i+conversion)
            array_down[i//conversion]=np.mean(array[i:i+conversion])
    return array_down

In [93]:
downsample1d(a, 3, 1)

array([1.1, 2.1, 3.1])

In [64]:
np.mean(a[0:3])

1.0999999999999999

In [53]:
a.rolling(3, center = True).mean()

0         NaN
1    1.100000
2    1.433333
3    1.766667
4    2.100000
5    2.433333
6    2.766667
7    3.100000
8         NaN
dtype: float64

## interp1d(a,np.arange(9))

In [26]:
resample(a,3)

array([1.76666667, 1.47353825, 3.05979508])

In [15]:
def moving_average(a, n=3) :
    ret = np.cumsum(a, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    return ret[n - 1:] / n

In [18]:
moving_average(a, 4)

array([1.325, 1.6  , 1.875, 2.325, 2.6  , 2.875])